# Поверхность сердца и её влияние на конечноэлементный расчёт

## Задача и объекты сравнения

Задача состоит из двух последовательных проверок. Сначала оценивается, можно ли превратить автоматическую маску MONAI Physio в технически пригодную поверхность. Затем на КТ Ника сравнивается электрический ответ одной и той же индивидуальной конечноэлементной модели (FEM), когда сердечная область задана отдельной фазовой маской или переносом исходной маски фазы 00.

Здесь важно различать три объекта. Объединение пяти высокодетальных классов `heartchambers_highres` включает миокард и четыре камеры; наружная маска `whole_heart` получена из задачи `total/heart`; FEM использует назначение сердечного материала тетраэдрам. Геометрический контроль первого объекта не доказывает пригодность второго и не предсказывает электрический ответ третьего. Все маски автоматические.

## Геометрический пилот на фазе 00 Ника

На исходной сетке сравнивались объединение меток `140–144` **полного** MONAI Physio и объединение тех же пяти проектных исходных масок TotalSegmentator v2 (TSv2). Обе поверхности построены одним методом marching cubes при `stride=1`, без морфологического закрытия или заполнения полостей. Поэтому их различие связано с входными масками, а не с разными алгоритмами построения поверхности. Номера фаз, аффинная матрица в координатах RAS и шаг сетки сверены; миллиметры приняты по происхождению КТ при единице NIfTI `unknown`. STL сам по себе единицы и координатную систему не хранит.

| Показатель | Полный MONAI Physio | Проектный TSv2 |
|---|---:|---:|
| Объём бинарной маски, мл | 569,387 | 571,987 |
| Вершины / треугольники | 129 894 / 259 788 | 130 524 / 261 052 |
| Площадь поверхности, мм² | 49 638,55 | 49 175,80 |
| Объём замкнутой поверхности, мл | 569,305 | 571,912 |
| Поверхностные компоненты | 4 | 1 |
| Характеристика Эйлера | 0 | −2 |

Обе бинарные маски 6-связны, а полученные поверхности замкнуты и имеют согласованную ориентацию граней. Четыре компоненты поверхности MONAI при одной компоненте воксельной маски указывают, что одной связности вокселей недостаточно для допуска к текущему маршруту тетраэдризации. Происхождение дополнительных компонент, в том числе возможные внутренние полости, не установлено. **Тетраэдризация нового объединения не запускалась.** Разница площадей **462,75 мм²** измерена при разных масках и не является разницей электрического ответа.

Очищенная проектная маска `whole_heart` — иной анатомический объект: у её поверхности **307 036** треугольников, площадь **67 206,76 мм²** и три поверхностные компоненты. Прежняя ошибка Gmsh для неё не переносится автоматически на объединение пяти классов. Протокол и локальные агрегаты приведены в [отчёте пилота](mp01_02_отчёт_пилота.md).

## Единая постановка FEM на КТ Ника

[Предварительная проверка входов](mp02_nix_fem_preflight.json) связала маски фаз 00, 01 и 07 с уже существующей индивидуальной FEM-моделью Ника по хэшам. Использованы одна подготовленная сетка (**791 141 узел, 4 448 426 тетраэдров**), одинаковый тканевый фон, конечные контакты полной электродной модели (CEM) диаметром **5 мм**, частота **50 кГц**. Существующий план охватывает 12 фаз и шесть монтажей. Это описание фиксирует конкретную исследовательскую постановку, а не универсальные параметры метода.

Статическая КТ и 4D-КТ имеют общий DICOM `FrameOfReferenceUID`; проверка внесердечных структур поддерживает направление преобразования координат, но не задаёт независимую верхнюю границу ошибки карты КТ→FEM. Верхнее касание маски `whole_heart` связано с меньшим охватом 4D-КТ по сравнению со статической и само по себе не блокирует сравнение в наблюдаемой области. [Проверка поля зрения](mp03_nix_fov_contact_report.md) и [отчёт о координатах](mp02_nix_paired_fem_report.md) сохраняют основания этого решения.

На этапе preflight зарегистрированная геометрия **не передавалась** в FEM: технически положительные поля 00→01 и сглаженное 00→07 ещё не имели независимого анатомического допуска. Позднее был выполнен отдельный **исследовательский парный расчёт**. Его наличие не меняет статус анатомической проверки.

## Регистрация фаз и допуск геометрии

Для всех **11 целевых фаз** Ника сохранены полноразрешённые сглаженные поля переноса маски 00. На узлах проверенной сетки у них не найдено неположительного определителя якобиана `det J`; расстояния до отдельных автоматических фазовых масок в среднем уменьшились. Положительный дискретный `det J` является необходимым техническим условием локальной ориентации поля на проверенных узлах. Он не исключает ошибку границы между узлами и не доказывает правильность анатомического движения. [Пофазный отчёт](mp03_nix_full_cycle_report.md) содержит параметры и отрицательные результаты.

Автор выбрал отдельные фазовые маски TSv2 как **автоматический контроль, независимый от переноса маски 00**. [Проверка происхождения](mp03_nix_ts2_reference_report.md) подтвердила фазу, сетку и совпадение хэшей с FEM-входами. Эти маски не являются принятой ручной анатомической истиной. Поэтому техническое прохождение контроля допускает **исследовательское сравнение чувствительности FEM**, но не использование перенесённого сердца как валидированной геометрии для функций сердца.

## Парное сравнение электрического ответа

На фазах **01, 03, 07 и 10** сопоставлены два варианта сердечной области: отдельно сегментированная фазовая маска TSv2 и маска фазы 00, перенесённая полем регистрации. Оба варианта имеют одну базовую фазу 00. Не менялись сетка, мягкотканный/лёгочный/костный фон, шесть монтажей TEPC-2…7, CEM-контакты диаметром 5 мм, проводимости, частота 50 кГц и решатель EIDORS/PARDISO. Материал сердечной области заново интегрирован на той же сетке. **24 сочетания фазы и монтажа** образуют парное сравнение; для перенесённых масок получены 24 новых решения, которые сопоставлены с ранее сохранёнными решениями для отдельных фазовых масок. Повторное интегрирование фазы 00 воспроизвело сохранённые доли тетраэдров побитово; проверены баланс токов и взаимность. Полные методы и QC — в [парном отчёте](mp02_nix_paired_fem_report.md).

Здесь `Z` — рассчитанный моделью импеданс в омах. Фазовый контраст `ΔZ = Z(фаза) − Z(00)` вычислен в **том же монтаже**; он не является измеренным сердечным сигналом. У вариантов общая база 00, поэтому разность их `ΔZ` совпадает с разностью `Z` целевой фазы.

| Фаза | Dice переноса к отдельной маске | Максимум разности двух `Z`, Ом | Максимум разности относительно `|ΔZ|` отдельной маски | Изменение знака `ΔZ` |
|---|---:|---:|---:|---|
| 01 | 0,94897 | 0,4666 | 54,1% | нет |
| 03 | 0,90099 | 0,7980 | 34,9% | нет |
| 07 | 0,95905 | 0,2046 | 125,9% | TEPC-5 |
| 10 | 0,96259 | 0,2691 | 108,2% | TEPC-2 и TEPC-5 |

По всем сочетаниям максимальная **абсолютная** разность `Z` составила **0,7980 Ом**; наибольшая относительная разность от абсолютного `Z` — **1,708%**. В трёх сочетаниях изменился знак модельного `ΔZ`. Относительные проценты для `ΔZ` велики также потому, что сам фазовый контраст может быть мал. Следовательно, близость абсолютного импеданса не гарантирует близости фазового сигнала. [Агрегат 24 решений](mp02_nix_paired_fem.json) и [проверка сохранённых входов и матриц](mp02_nix_paired_fem_verification.json) позволяют проследить расчёт. Позднейшая сверка TSv2 повторно прочитала те же решения и подтвердила хэши; это **не новый независимый FEM-эксперимент**.

## Уровень доказательности и условия дальнейшего применения

Отдельные фазовые маски TSv2 получены без использования полей переноса, а их хэши совпадают с масками парного FEM-опыта. Это обеспечивает **независимость автоматического компаратора от регистрации** и прослеживаемость расчёта. Анатомическая точность обоих вариантов сердечной области относительно реального сердца Ника остаётся неизвестной. Не выполнены также сеточная сходимость парного эффекта, проверка альтернативных проводимостей и контактов и сопоставление с одновременно измеренным ТТРКГ.

Парный опыт показывает **чувствительность конкретной электрической модели к способу задания геометрии**. Изменение знака `ΔZ` — отрицательный результат для предположения, что небольшая разница абсолютного `Z` всегда безопасна. Он не определяет, какой вариант геометрии ближе к анатомии, и не даёт численной ошибки УО или ФВ.

Для проверки пригодности к количественной методике нужны принятый независимый анатомический критерий соответствующих фаз либо явно обоснованная альтернатива при слабом контрасте, оценка ошибки карты КТ→FEM, контроль топологии и сеточной сходимости, а затем повторное FEM-сравнение на допустимых геометриях и параметрах. Касание верхнего среза 4D-КТ само по себе в этот перечень блокеров не входит.

In [ ]:
from datetime import datetime, timezone
import os, subprocess, sys
from pathlib import Path
mask = os.environ.get('MP02_MASK')
if not mask:
    print('Set MP02_MASK to a local binary NIfTI mask')
else:
    output = Path('runs') / f"mp01_02_geometry_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json"
    command = [sys.executable, 'mp01_02_pilot.py', 'geometry', '--mask', mask, '--stride', os.environ.get('MP02_STRIDE', '1'), '--output', str(output)]
    if os.environ.get('MP02_VERIFIED_MM') == '1': command.append('--assume-mm')
    if os.environ.get('MP02_REFERENCE_STL'): command.extend(['--reference-surface', os.environ['MP02_REFERENCE_STL']])
    if os.environ.get('MP02_TETRA') == '1': command.append('--tetra')
    subprocess.run(command, check=True)


In [ ]:
# Reproduce the paired highres geometry QC using local paths only.
# Set MP02_CT, MP02_BASELINE_HIGHRES_DIR and MP02_FULL_CANDIDATE_DIR locally.
import hashlib, json
from datetime import datetime, timezone
import nibabel as nib
import numpy as np
from mp01_02_compare_heart import LABELS, binary, sha256
keys = ('MP02_CT', 'MP02_BASELINE_HIGHRES_DIR', 'MP02_FULL_CANDIDATE_DIR')
missing = [key for key in keys if not os.environ.get(key)]
if missing:
    print('Local inputs not configured:', ', '.join(missing))
else:
    ct_path = Path(os.environ['MP02_CT'])
    candidate_dir = Path(os.environ['MP02_FULL_CANDIDATE_DIR'])
    manifest = json.loads((candidate_dir / 'mp01_02_manifest.json').read_text(encoding='utf-8'))
    assert manifest['full_monai_physio_workflow'] is True
    assert manifest['executed_tasks'] == ['total', 'heartchambers_highres', 'tissue_4_types', 'lung_vessels', 'body']
    assert manifest['ct_sha256'] == sha256(ct_path)
    ct = nib.load(str(ct_path))
    baseline_union = np.zeros(ct.shape, dtype=bool)
    for filename in LABELS.values():
        baseline_union |= binary(Path(os.environ['MP02_BASELINE_HIGHRES_DIR']) / filename, ct)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    root = Path('runs').resolve()
    baseline_path = root / f'mp01_02_project_highres_union_{stamp}.nii.gz'
    assert not baseline_path.exists()
    image = nib.Nifti1Image(baseline_union.astype(np.uint8), ct.affine, ct.header)
    image.set_data_dtype(np.uint8)
    nib.save(image, str(baseline_path))
    for route, mask_path in [('project', baseline_path), ('full_monai', candidate_dir / 'mp01_02_highres_heart_union.nii.gz')]:
        output = root / f'mp01_02_{route}_highres_geometry_{stamp}.json'
        subprocess.run([sys.executable, 'mp01_02_pilot.py', 'geometry', '--mask', str(mask_path), '--stride', '1', '--assume-mm', '--output', str(output)], check=True)
        result = json.loads(output.read_text(encoding='utf-8'))
        print(route, result['original_mask'], result['surface'])
    # Run Gmsh separately only after verifying one valid connected boundary.


## Воспроизведение и ограничения геометрической подготовки

Кодовые ячейки этого ноутбука содержат команды для повторения локального контроля геометрии при заданных пользователем путях; первичные КТ, STL, объёмные сетки и индивидуальные поля хранятся вне Git или в исключённом `runs/`. Переход к объёмной сетке и электрическому решению нельзя заменить простым экспортом STL. Морфологическое закрытие, заполнение полостей и `stride>1` меняют представляемую анатомию; их допустимо изучать только как отдельно подписанные сценарии с новым QC.

Геометрический пилот по объединению пяти высокодетальных классов **не включал** тетраэдризацию и расчёт `Z/ΔZ`. Парный FEM-опыт выше относится к другой, автоматической фазовой маске `whole_heart` Ника и к перенесённому варианту той же сердечной области. Эти результаты нельзя объединять в одно сравнение алгоритмов построения поверхности. Числа в таблицах следует читать вместе с соответствующим отчётом и происхождением данных; кодовые ячейки в этой редакции не исполнялись.